In [24]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from ucimlrepo import fetch_ucirepo
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.compose import TransformedTargetRegressor
import matplotlib.pyplot as plt
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
    r2_score
)

In [18]:
df = pd.read_excel("RENDIMIENTO.xlsx")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 752 entries, 0 to 751
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Nacional      752 non-null    int64  
 1   Grado         752 non-null    int64  
 2   Matricula     752 non-null    int64  
 3   Convocatoria  752 non-null    int64  
 4   EP2           752 non-null    float64
 5   EP1           752 non-null    float64
 6   EF            752 non-null    float64
 7   NF            752 non-null    float64
dtypes: float64(4), int64(4)
memory usage: 47.1 KB


In [19]:
df.groupby("Nacional").describe().T

Nacional                     0           1
Grado        count  449.000000  303.000000
             mean     2.244989    2.752475
             std      1.238385    1.398244
             min      1.000000    1.000000
             25%      1.000000    1.000000
             50%      3.000000    3.000000
             75%      3.000000    3.000000
             max      5.000000    5.000000
Matricula    count  449.000000  303.000000
             mean     1.414254    1.151815
             std      0.760270    0.498396
             min      1.000000    1.000000
             25%      1.000000    1.000000
             50%      1.000000    1.000000
             75%      2.000000    1.000000
             max      5.000000    4.000000
Convocatoria count  449.000000  303.000000
             mean     0.385301    0.234323
             std      0.487209    0.424276
             min      0.000000    0.000000
             25%      0.000000    0.000000
             50%      0.000000    0.000000
             75%      1.000000    0.000000
             max      1.000000    1.000000
EP2          count  449.000000  303.000000
             mean     4.145880    5.149835
             std      3.692740    3.621888
             min      0.000000    0.000000
             25%      0.500000    1.300000
             50%      3.500000    5.400000
             75%      7.700000    8.700000
             max     10.000000   10.000000
EP1          count  449.000000  303.000000
             mean     2.003118    4.004620
             std      2.236465    2.726904
             min      0.000000    0.000000
             25%      0.000000    1.650000
             50%      1.200000    3.800000
             75%      3.200000    6.300000
             max     10.000000   10.000000
EF           count  449.000000  303.000000
             mean     2.545212    4.754785
             std      2.889062    3.009824
             min      0.000000    0.000000
             25%      0.000000    2.500000
             50%      1.500000    5.000000
             75%      5.000000    7.000000
             max     10.000000   10.000000
NF           count  449.000000  303.000000
             mean     2.451670    4.653795
             std      2.724148    2.898808
             min      0.000000    0.000000
             25%      0.000000    2.500000
             50%      1.500000    5.200000
             75%      5.000000    6.900000
             max      9.900000   10.000000

In [20]:
X = df.drop(columns=["EF", "NF"])
y = df["EF"].values.ravel()

X.info()

<class 'pandas.DataFrame'>
RangeIndex: 752 entries, 0 to 751
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Nacional      752 non-null    int64  
 1   Grado         752 non-null    int64  
 2   Matricula     752 non-null    int64  
 3   Convocatoria  752 non-null    int64  
 4   EP2           752 non-null    float64
 5   EP1           752 non-null    float64
dtypes: float64(2), int64(4)
memory usage: 35.4 KB


In [21]:
# Supongamos X (características) e y (variable continua)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42
)

In [ ]:
# 1. Definición de los Pipelines (Esencial escalar variables continuas previa regularización)
pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=0.1, random_state=42))
])

pipe_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Lasso(alpha=0.1, random_state=42))
])

pipe_elastic = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42))
])

pipe_knn_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", KNeighborsRegressor(n_neighbors=5, weights="distance", p=2))
])

Pipelines = [
    ("Ridge", pipe_ridge),
    ("Lasso", pipe_lasso),
    ("ElasticNet", pipe_elastic)

]

for pipe_name, pipeline in Pipelines:

    pipeline.fit(X_train, y_train)
    coeficientes = pipeline.named_steps["regressor"].coef_
    intercepto = pipeline.named_steps["regressor"].intercept_
    df_coef = pd.DataFrame({
    "Variable": X_train.columns,
    "Coeficiente": coeficientes
    }).sort_values(by="Coeficiente", key=abs, ascending=False)
    y_pred = pipeline.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    print("Resultados del Pipeline: ", pipe_name)
    print(f"R^2: {r2:.3f}")
    print(f"Intercepto (Beta_0): {intercepto:.4f}\n")
    print(df_coef)

Resultados del Pipeline:  Ridge
R^2: 0.481
Intercepto (Beta_0): 3.4379

       Variable  Coeficiente
5           EP1     1.138794
4           EP2     1.104502
0      Nacional     0.484041
3  Convocatoria    -0.302493
2     Matricula     0.106070
1         Grado    -0.054806
Resultados del Pipeline:  Lasso
R^2: 0.485
Intercepto (Beta_0): 3.4379

       Variable  Coeficiente
5           EP1     1.096733
4           EP2     1.026373
0      Nacional     0.388621
3  Convocatoria    -0.235341
1         Grado    -0.000000
2     Matricula     0.000000
Resultados del Pipeline:  ElasticNet
R^2: 0.486
Intercepto (Beta_0): 3.4379

       Variable  Coeficiente
5           EP1     1.072663
4           EP2     1.022021
0      Nacional     0.421577
3  Convocatoria    -0.274546
2     Matricula     0.005938
1         Grado    -0.000000


In [30]:
pipe_knn_reg.fit(X_train, y_train)

# Predicción continua
y_pred = pipe_knn_reg.predict(X_test)

# Variabilidad Explicada
r2 = r2_score(y_test, y_pred)

print(r2)

0.5226232693701051


In [36]:
# 1. Definición de los Pipelines (Esencial escalar variables continuas previa regularización)
pipe_ridge = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=0.1, random_state=42))
])

pipe_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Lasso(alpha=0.1, random_state=42))
])

pipe_elastic = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=42))
])

pipe_knn_reg = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", KNeighborsRegressor(n_neighbors=5, weights="distance", p=2))
])

modelo_arbol_reg = DecisionTreeRegressor(
    criterion="squared_error",  # Alternativas: 'absolute_error', 'friedman_mse'
    max_depth=5,
    min_samples_split=10,
    min_samples_leaf=4,
    random_state=42
)

# Definición del Pipeline
modelo_rf_reg = RandomForestRegressor(
    n_estimators=100,
    max_features=11,     # En regresión se suele evaluar un porcentaje mayor de características
    max_depth=8,
    random_state=42,
    n_jobs=-1
)

modelo_gb_reg = HistGradientBoostingRegressor(
    loss="squared_error",   # Alternativas: 'absolute_error', 'huber' (robusto a outliers)
    learning_rate=0.05,
    max_iter=100,
    max_depth=6,
    random_state=42
)

pipe_svr_rbf = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", SVR(kernel="rbf", C=10.0, epsilon=0.1, gamma="scale"))
])

Pipelines = [
    ("Ridge", pipe_ridge),
    ("Lasso", pipe_lasso),
    ("ElasticNet", pipe_elastic),
    ("Regresor KNN", pipe_knn_reg),
    ("Arbol", modelo_arbol_reg),
    ("RandomForestRegressor", modelo_rf_reg),
    ("HistGradientBoostingRegressor", modelo_gb_reg),
    ("SVR", pipe_svr_rbf)

]

for pipe_name, pipeline in Pipelines:

    pipeline.fit(X_train, y_train)
   
    y_pred = pipeline.predict(X_test)

    r2 = r2_score(y_test, y_pred)

    print("Resultados del Pipeline: ", pipe_name)
    print(f"R^2: {r2:.3f}")

Resultados del Pipeline:  Ridge
R^2: 0.481
Resultados del Pipeline:  Lasso
R^2: 0.485
Resultados del Pipeline:  ElasticNet
R^2: 0.486
Resultados del Pipeline:  Regresor KNN
R^2: 0.523
Resultados del Pipeline:  Arbol
R^2: 0.459
Resultados del Pipeline:  RandomForestRegressor
R^2: 0.525
Resultados del Pipeline:  HistGradientBoostingRegressor
R^2: 0.494
Resultados del Pipeline:  SVR
R^2: 0.532


In [42]:
# Pipeline base
pipe_svr = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", SVR())
])

# Rejilla de hiperparámetros (sintaxis paso__parametro)
param_grid = {
    "regressor__C": [0.1, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0],
    "regressor__epsilon": [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
    "regressor__kernel": ["rbf", "poly"]
}

# 3. K-Fold estándar para regresión
cv_regresion = KFold(n_splits=5, shuffle=True, random_state=42)

# 4. GridSearchCV optimizando por RMSE (neg_root_mean_squared_error)
grid_svr = GridSearchCV(
    estimator=pipe_svr_rbf,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv_regresion,
    n_jobs=-1
)

grid_svr.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_svr.best_params_}")
print(f"Mejor RMSE (CV): {-grid_svr.best_score_:.4f}")

Mejores parámetros: {'regressor__C': 1.5, 'regressor__epsilon': 1.0, 'regressor__kernel': 'rbf'}
Mejor RMSE (CV): 2.2250


In [44]:
# Pipeline base
modelo_rf_reg = RandomForestRegressor()

# Rejilla de hiperparámetros (sintaxis paso__parametro)
param_grid = {
    "max_depth": [4, 6, 8, 10, None],
    "min_samples_split": [2, 4, 8, 10],
    "min_samples_leaf": [3, 5, 7, 9],
}

# 3. K-Fold estándar para regresión
cv_regresion = KFold(n_splits=5, shuffle=True, random_state=42)

# 4. GridSearchCV optimizando por RMSE (neg_root_mean_squared_error)
grid_rf = GridSearchCV(
    estimator=modelo_rf_reg,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv_regresion,
    n_jobs=-1
)

grid_rf.fit(X_train, y_train)

print(f"Mejores parámetros: {grid_rf.best_params_}")
print(f"Mejor RMSE (CV): {-grid_rf.best_score_:.4f}")

Mejores parámetros: {'max_depth': 8, 'min_samples_leaf': 7, 'min_samples_split': 8}
Mejor RMSE (CV): 2.2155


In [45]:
mejor_modelo = grid_rf.best_estimator_

df["EF_pred"] = mejor_modelo.predict[X]

TypeError: 'method' object is not subscriptable